In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
file = "PARTES_DE_ACCIDENTE_2020_25.xlsx"

xls = pd.ExcelFile(file)

print(xls.sheet_names)

['2020', '2021', '2022', '2023', '2024', '2025']


In [3]:
dfs = pd.read_excel(file, sheet_name=None)

In [4]:
dfs.keys() # Verifico que tengo los nombres de todas las hojas 

dict_keys(['2020', '2021', '2022', '2023', '2024', '2025'])

In [5]:
df = pd.concat(
    [d.assign(sheet=name) for name, d in dfs.items()],
    ignore_index=True
) # Las concateno en un solo DF

In [ ]:
df.head()

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3982 entries, 0 to 3981
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   NIF                    3982 non-null   object        
 1   Nombre                 3982 non-null   object        
 2   Apellido1              3982 non-null   object        
 3   Apellido2              3981 non-null   object        
 4   FechaParteAccidente    3982 non-null   datetime64[ns]
 5   FechaAltaParte         3982 non-null   datetime64[ns]
 6   Codigo                 3982 non-null   int64         
 7   DescripcionParte       3982 non-null   object        
 8   Actividad              3982 non-null   object        
 9   Lugar                  3860 non-null   object        
 10  FechaRecepcion         3982 non-null   datetime64[ns]
 11  CentroUrgencias        1604 non-null   object        
 12  Helicoptero            2 non-null      object        
 13  Mac

In [9]:
cols_keep = [
    "NIF",
    "FechaParteAccidente",
    "Actividad",
    "Lugar",
    "Provincia",
    "TipoAccidente",
    "DescripcionGrado",
    "TamañoGrupo",
    "NResponsables",
    "Entrenamiento",
    "ActividadPersonal",
    "ActividadOrganizada",
    "Festivo",
    "sheet"
]

df = df[cols_keep] # Las columnas que creo que nos pueden aportar información relevante para el análisis

In [ ]:
df.head()

In [11]:
df.info() # Nos quedamos con 14 de 26 columnas

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3982 entries, 0 to 3981
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   NIF                  3982 non-null   object        
 1   FechaParteAccidente  3982 non-null   datetime64[ns]
 2   Actividad            3982 non-null   object        
 3   Lugar                3860 non-null   object        
 4   Provincia            3982 non-null   object        
 5   TipoAccidente        3982 non-null   object        
 6   DescripcionGrado     3830 non-null   object        
 7   TamañoGrupo          3982 non-null   int64         
 8   NResponsables        3982 non-null   int64         
 9   Entrenamiento        3982 non-null   bool          
 10  ActividadPersonal    3982 non-null   bool          
 11  ActividadOrganizada  3982 non-null   bool          
 12  Festivo              3982 non-null   bool          
 13  sheet                3982 non-nul

In [12]:
df.duplicated().sum()

np.int64(8)

In [ ]:
dups = df[df.duplicated(
    subset=[
        "NIF",
        "FechaParteAccidente",
        "Lugar",
        "TipoAccidente"
    ],
    keep=False
)]

dups # Muestra los registros duplicados - los originales y las copias

In [14]:
dups[["NIF", "FechaParteAccidente", "sheet"]].sort_values(
    ["NIF", "FechaParteAccidente"]
) # Muestra los registros duplicados por hoja - doble comprobación de que son iguales

,NIF,FechaParteAccidente,sheet
3276,02282680E,2024-09-27 21:00:00,2024
3277,02282680E,2024-09-27 21:00:00,2024
2826,02302403B,2024-01-18 20:15:00,2024
2827,02302403B,2024-01-18 20:15:00,2024
3133,02442123Y,2024-06-25 16:00:00,2024
3134,02442123Y,2024-06-25 16:00:00,2024
3534,02652024D,2025-03-02 16:00:00,2025
3535,02652024D,2025-03-02 16:00:00,2025
2328,02665694V,2023-04-11 00:00:00,2023
2329,02665694V,2023-04-11 00:00:00,2023


In [15]:
df = df.drop_duplicates(
    subset=[
        "NIF",
        "FechaParteAccidente",
        "sheet"
    ]
) # Elimino los registros duplicados, manteniendo solo uno de ellos (el original)

In [16]:
df.duplicated(
    subset=[
        "NIF",
        "FechaParteAccidente",
        "sheet"
    ]
).sum() # Verificación de que ya no hay registros duplicados en el DataFrame final

np.int64(0)

In [17]:
df["NIF"] = df["NIF"].astype(str).str.strip().str.upper() # Normalizo el formato de los NIF para evitar problemas de merge

In [18]:
df["NIF"].isna().sum(), (df["NIF"]=="").sum() # Todos los NIF están completos, no hay valores nulos ni vacíos!

(np.int64(0), np.int64(0))

In [19]:
df["FechaParteAccidente"].isna().sum()

np.int64(0)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3957 entries, 0 to 3981
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   NIF                  3957 non-null   object        
 1   FechaParteAccidente  3957 non-null   datetime64[ns]
 2   Actividad            3957 non-null   object        
 3   Lugar                3836 non-null   object        
 4   Provincia            3957 non-null   object        
 5   TipoAccidente        3957 non-null   object        
 6   DescripcionGrado     3805 non-null   object        
 7   TamañoGrupo          3957 non-null   int64         
 8   NResponsables        3957 non-null   int64         
 9   Entrenamiento        3957 non-null   bool          
 10  ActividadPersonal    3957 non-null   bool          
 11  ActividadOrganizada  3957 non-null   bool          
 12  Festivo              3957 non-null   bool          
 13  sheet                3957 non-null   o

In [ ]:
df

In [ ]:
# Cambio el nombre de sheet por año de accidente y borro fecha parte de accidente
df = df.drop("FechaParteAccidente", axis=1)
df = df.rename(columns={"sheet": "año_accidente"})
df

In [23]:
df.to_csv('accidentes_ricardo.csv', index=False)